# Book Recommender System

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict
import json
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from surprise import Dataset, Reader, KNNWithMeans, SVD
from surprise import accuracy
from sklearn.metrics import mean_absolute_error, mean_squared_error

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tranm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tranm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Data Preparation and Preprocessing

https://cseweb.ucsd.edu/~jmcauley/datasets/goodreads.html

In [2]:
genre = "poetry"

In [3]:
books_data = []
with open(f'{genre}_books.json', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        languages = item.get('language_code')
        if 'en' in languages and item.get('description', '').strip():
            shelves = [shelf['name'] for shelf in item.get('popular_shelves', []) if int(shelf['count']) > 1]
            shelves_str = " ".join(shelves)
            
            books_data.append({
                'book_id': item['book_id'],
                'title': item['title'],
                'author_id': item['authors'][0]['author_id'] if item['authors'] else None,
                'description': item['description'],
                'tags': shelves_str,
                'average_rating': float(item['average_rating']) if item['average_rating'] else 0.0,
                'ratings_count': int(item['ratings_count']) if item['ratings_count'] else 0
            })

In [4]:
books_df = pd.DataFrame(books_data).drop_duplicates(subset=['book_id'])
books_df

,book_id,title,author_id,description,tags,average_rating,ratings_count
0,16037549,Vision of Sir Launfal and Other Poems,15585,Number 30 in a series of literary pamphlets pu...,to-read poetry currently-reading,3.83,3
1,29065952,Louder Than Everything You Love,14308759,Louder Than Everything You Love is about trans...,to-read poetry,5.00,9
2,15861988,Into Temptation,2988946,Into Temptation is the debut collection of poe...,to-read poetry,4.75,8
3,24849837,Naked Soul: The Erotic Love Poems,13260036,"""Erotic poetry that evokes feelings of joy, ha...",to-read poetry currently-reading first-reads,3.95,27
4,17729612,The More Loving One,14002590,Free online poetry.\nFrom Random House's Boldt...,to-read poetry 51-of-the-most-beautiful-sentences,4.00,13
...,...,...,...,...,...,...,...
8256,13452060,Ramblings & Rhymes: An Anthology of Poetry,5624502,An anthology of poetry showing snap shots Arie...,to-read pagan-poetry books-i-have-written poet...,4.55,10
8257,11066682,"Field Work: Notes, Songs, Poems 1997-2010",1268000,"Poetry. ""In San Francisco, Austin and Buffalo ...",to-read poetry,4.89,9
8258,28146064,Straight James/Gay James,3369521,Actor James Franco's chapbook of poems explore...,to-read poetry currently-reading netgalley lgb...,3.42,219
8259,28923921,Call Me by My Other Name,4642310,"Call Me by My Other Name is a fierce, unforget...",to-read poetry lgbtqia,4.28,17


In [5]:
interactions_data = []
with open(f'{genre}_interactions.json', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        if int(item['rating']) > 0:
            interactions_data.append({
                'user_id': item['user_id'],
                'book_id': item['book_id'],
                'review_id': item['review_id'],
                'rating': int(item['rating']),
                'is_read': item['is_read']
            })

In [6]:
interactions_df = pd.DataFrame(interactions_data)
interactions_df = interactions_df[interactions_df['book_id'].isin(books_df['book_id'])]
interactions_df

,user_id,book_id,review_id,rating,is_read
0,8842281e1d1347389f2ab93d60773d4d,1384,1bad0122cebb4aa9213f9fe1aa281f66,4,True
1,8842281e1d1347389f2ab93d60773d4d,1376,eb6e502d0c04d57b43a5a02c21b64ab4,4,True
2,8842281e1d1347389f2ab93d60773d4d,30119,787564bef16cb1f43e0f641ab59d25b7,5,True
3,72fb0d0087d28c832f15776b0d936598,30119,2a83589fb597309934ec9b1db5876aaf,3,True
4,ab2923b738ea3082f5f3efcbbfacb218,240007,149b57966342c0c99eb8ec234acb7cdb,4,True
...,...,...,...,...,...
1229038,19f9994e793d391cc36043801012ca0f,18263725,31c05a575e619c8c87e65c12a9977bf7,3,True
1229043,bfc558b791304f0ce74ad1c3a6ab08f7,2696,860a81c2fa2ae9276361bb8da2c4bd4b,3,True
1229044,bfc558b791304f0ce74ad1c3a6ab08f7,1371,641f69bd80baba71c2b9ca2f0e1a34bc,4,True
1229045,bfc558b791304f0ce74ad1c3a6ab08f7,1381,789d7b3ab24311bccd5ed9371e13b2c5,5,True


In [ ]:
# min = 3

# user_counts = interactions_df['user_id'].value_counts()
# active_users = user_counts[user_counts >= min].index
# interactions_df = interactions_df[interactions_df['user_id'].isin(active_users)]

# book_counts = interactions_df['book_id'].value_counts()
# popular_books = book_counts[book_counts >= min].index
# interactions_df = interactions_df[interactions_df['book_id'].isin(popular_books)]

# books_df = books_df[books_df['book_id'].isin(popular_books)]

In [7]:
interactions_train, interactions_test = train_test_split(
    interactions_df, 
    test_size=0.2, 
    random_state=42
)

In [8]:
reviews_data = []
with open(f'{genre}_reviews.json', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        if item.get('review_text', '').strip():
            reviews_data.append({
                'review_id': item['review_id'],
                'user_id': item['user_id'],
                'book_id': item['book_id'],
                'review_text': item['review_text']
            })

reviews_df = pd.DataFrame(reviews_data)

train_review_ids = set(interactions_train['review_id'].dropna())
reviews_train = reviews_df[reviews_df['review_id'].isin(train_review_ids)]

In [12]:
reviews_train

,review_id,user_id,book_id,review_text
0,28423ff309bc896c071a8d9df4a10e8a,3ca7375dba942a760e53b726c472a7dd,402128,I have three younger siblings and we grew up w...
3,cb1ebc02d8b2aff15735d513877463ce,d37b46b2190ed7c518259f29b47a9b36,253264,I just reread this play for a class I am takin...
4,8dca128b8e869048a7442c18659dbece,af157d0205b8a901dee6d4a2aed7e6ad,70885,"Cuanto mas leo, mas me gusta. Su poesia es env..."
6,e1705ca7a772b08edf9b3940d7b1c088,af157d0205b8a901dee6d4a2aed7e6ad,23534,"Poemas crudos y reales, sin tanto adorno aun u..."
7,22a588275798c15ed7757db0c526e296,f4c6fe33ef61c38f7f4aeb5224c259a5,13105527,"This ain't a book with to die for characters, ..."
...,...,...,...,...
154374,49dcc99043fc89b9365b29ae83353f03,a5ce12010d33bc18359538ec21c2c763,102331,This is like my 2nd soulmate beside coffee. I ...
154379,1bcebd49636cde3ac606a5be73143ab5,52d58d3b524bf97067792e273f9ba9fa,195769,"At times difficult to hold interest, but amazi..."
154380,7be3e8317a9971e6e5499d096b74891c,52d58d3b524bf97067792e273f9ba9fa,1371,What a difference decades make. I read this in...
154386,274798a3e30dad46023cbc12a8a870f5,6bfd86577e73d3d0709451bdc788a501,18003300,<3 I LOVE IT!!!


In [14]:
def clean_text(text):
    if not text:
        return ""
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    cleaned_tokens = [word for word in tokens if word not in stop_words and word.isalpha()]
    return " ".join(cleaned_tokens)

books_df['cleaned_content'] = (books_df['description'] + " " + books_df['tags']).apply(clean_text)
reviews_train['cleaned_review'] = reviews_train['review_text'].apply(clean_text)

user_profiles = reviews_train.groupby('user_id')['cleaned_review'].apply(lambda x: " ".join(x)).reset_index()

In [15]:
books_df

,book_id,title,author_id,description,tags,average_rating,ratings_count,cleaned_content
0,16037549,Vision of Sir Launfal and Other Poems,15585,Number 30 in a series of literary pamphlets pu...,to-read poetry currently-reading,3.83,3,number series literary pamphlets published mon...
1,29065952,Louder Than Everything You Love,14308759,Louder Than Everything You Love is about trans...,to-read poetry,5.00,9,louder everything love transformation narrator...
2,15861988,Into Temptation,2988946,Into Temptation is the debut collection of poe...,to-read poetry,4.75,8,temptation debut collection poems sophia black...
3,24849837,Naked Soul: The Erotic Love Poems,13260036,"""Erotic poetry that evokes feelings of joy, ha...",to-read poetry currently-reading first-reads,3.95,27,erotic poetry evokes feelings joy happiness ov...
4,17729612,The More Loving One,14002590,Free online poetry.\nFrom Random House's Boldt...,to-read poetry 51-of-the-most-beautiful-sentences,4.00,13,free online poetry random houses boldtype exce...
...,...,...,...,...,...,...,...,...
8256,13452060,Ramblings & Rhymes: An Anthology of Poetry,5624502,An anthology of poetry showing snap shots Arie...,to-read pagan-poetry books-i-have-written poet...,4.55,10,anthology poetry showing snap shots ariettas h...
8257,11066682,"Field Work: Notes, Songs, Poems 1997-2010",1268000,"Poetry. ""In San Francisco, Austin and Buffalo ...",to-read poetry,4.89,9,poetry san francisco austin buffalo chiels amo...
8258,28146064,Straight James/Gay James,3369521,Actor James Franco's chapbook of poems explore...,to-read poetry currently-reading netgalley lgb...,3.42,219,actor james francos chapbook poems explores di...
8259,28923921,Call Me by My Other Name,4642310,"Call Me by My Other Name is a fierce, unforget...",to-read poetry lgbtqia,4.28,17,call name fierce unforgettable book bodies des...


In [16]:
reviews_train

,review_id,user_id,book_id,review_text,cleaned_review
0,28423ff309bc896c071a8d9df4a10e8a,3ca7375dba942a760e53b726c472a7dd,402128,I have three younger siblings and we grew up w...,three younger siblings grew watching musical c...
3,cb1ebc02d8b2aff15735d513877463ce,d37b46b2190ed7c518259f29b47a9b36,253264,I just reread this play for a class I am takin...,reread play class taking scopes trial took pla...
4,8dca128b8e869048a7442c18659dbece,af157d0205b8a901dee6d4a2aed7e6ad,70885,"Cuanto mas leo, mas me gusta. Su poesia es env...",cuanto mas leo mas gusta su poesia es envolvente
6,e1705ca7a772b08edf9b3940d7b1c088,af157d0205b8a901dee6d4a2aed7e6ad,23534,"Poemas crudos y reales, sin tanto adorno aun u...",poemas crudos reales sin tanto adorno aun usan...
7,22a588275798c15ed7757db0c526e296,f4c6fe33ef61c38f7f4aeb5224c259a5,13105527,"This ain't a book with to die for characters, ...",aint book die characters impeccably romantic s...
...,...,...,...,...,...
154374,49dcc99043fc89b9365b29ae83353f03,a5ce12010d33bc18359538ec21c2c763,102331,This is like my 2nd soulmate beside coffee. I ...,like soulmate beside coffee brought anywhere b...
154379,1bcebd49636cde3ac606a5be73143ab5,52d58d3b524bf97067792e273f9ba9fa,195769,"At times difficult to hold interest, but amazi...",times difficult hold interest amazing see anal...
154380,7be3e8317a9971e6e5499d096b74891c,52d58d3b524bf97067792e273f9ba9fa,1371,What a difference decades make. I read this in...,difference decades make read high school colle...
154386,274798a3e30dad46023cbc12a8a870f5,6bfd86577e73d3d0709451bdc788a501,18003300,<3 I LOVE IT!!!,love


In [17]:
user_profiles

,user_id,cleaned_review
0,0004ae25e3cf5f5a44b6f1ccfdd3d343,n hbbt hdh lktb wwjdt fyh kthr mn qs hb hy qs ...
1,0006a5b8cda1ba6d7b911dc575f6547b,nu chto skazat v kotoryi raz ubedilsia chto pu...
2,000a1016fda6008d1edbba720ca00851,poetry need long need contain rhyming words co...
3,001010815d3b2692435dfc70285d06e4,poem aint lesser known title equally good book...
4,00199435f2a5dc362c793ea91964bfe2,honestly dont read much poetry ive feeling nee...
...,...,...
24411,ffefaa26fa788acacf550a7e047e2370,great book use introducing poetry first timers...
24412,fff07bab21bcf970b61f5c7a642fade7,ive read times remains one favorite pieces lit...
24413,fff7f660fc5277b7c544dd57dbbf95f3,author kindly sent free copy exchange honest r...
24414,fffcf6da0f39d7ab624e2a8da054d2c3,beautiful ache


In [19]:
print(f"Активный репозиторий книг: {books_df.shape[0]} шт.")
print(f"Матрица обучения: {interactions_train.shape[0]} взаимодействий.")
print(f"Матрица тестирования: {interactions_test.shape[0]} взаимодействий.")
print(f"Сгенерировано изолированных профилей пользователей: {user_profiles.shape[0]} шт.")

Активный репозиторий книг: 8261 шт.
Матрица обучения: 567891 взаимодействий.
Матрица тестирования: 141973 взаимодействий.
Сгенерировано изолированных профилей пользователей: 24416 шт.


In [66]:
reviewers = set(user_profiles['user_id'])
active_reviewers = interactions_train[interactions_train['user_id'].isin(reviewers)]
reviewers_counts = active_reviewers['user_id'].value_counts()
# train_user_counts = interactions_train['user_id'].value_counts()

# User with the most books read in the training set
active_user_id = reviewers_counts.index[0]
active_user_count = reviewers_counts.iloc[0]

# User with 0 interactions in training set (Cold start)
train_users = set(interactions_train['user_id'].unique())
test_users = set(interactions_test['user_id'].unique())
cold_users = list(test_users - train_users)

if len(cold_users) > 0:
    cold_user_id = cold_users[0]
    cold_user_count = 0
else:
    # If no true 0-read users exist, pick someone with least books read
    cold_user_id = reviewers_counts.index[-1]
    cold_user_count = reviewers_counts.iloc[-1]

print(f"Active User:\t{active_user_id} ({active_user_count} reads)")
print(f"Cold User:\t{cold_user_id} ({cold_user_count} reads)")

Active User:	23b7c6868d2dfe394b8395e86398eb26 (262 reads)
Cold User:	e4b4381c5269d210f225ad4ed97f85db (0 reads)


In [ ]:
def get_matched_books(user_id, recommendations_df):
    user_test_profile = interactions_test[interactions_test['user_id'] == user_id]
    actual_book_ids = set(user_test_profile['book_id'].values)

    predicted_book_ids = set(recommendations_df['book_id'].values)
    hits = predicted_book_ids.intersection(actual_book_ids)
    hits_df = recommendations_df[recommendations_df['book_id'].isin(hits)].reset_index()

    merged_df = hits_df.merge(user_test_profile[['book_id', 'rating', 'is_read']], on='book_id', how='inner')
    final_df = merged_df.set_index('index')
    final_df.index.name = recommendations_df.index.name
    
    return final_df

## Content-Based Filtering

In [ ]:
# 1. Инициализация векторизатора TF-IDF
# Ограничиваем словарь до 5000 самых важных слов для экономии памяти и ускорения расчетов
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# 2. Обучение векторизатора на описаниях и тегах книг
tfidf_book_matrix = tfidf.fit_transform(books_df['cleaned_content'])
book_id_to_idx = {book_id: idx for idx, book_id in enumerate(books_df['book_id'])}

# 3. Проекция профилей пользователей в то же векторное пространство
tfidf_user_matrix = tfidf.transform(user_profiles['cleaned_review'])
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(user_profiles['user_id'])}

# 4. Функция генерации рекомендаций на основе косинусного сходства
def get_cbf_recommendations(user_id, top_n=10):
    if user_id not in user_id_to_idx:
        # Если пользователя нет в текстовых профилях (не писал отзывы) — возвращаем пустой список
        return []
    
    # Извлекаем вектор конкретного пользователя
    user_idx = user_id_to_idx[user_id]
    user_vector = tfidf_user_matrix[user_idx]
    
    # Вычисляем сходство этого пользователя со ВСЕМИ книгами в базе данных
    similarity_scores = cosine_similarity(user_vector, tfidf_book_matrix).flatten()
    
    # Сортируем индексы книг по убыванию оценки сходства
    top_book_indices = np.argsort(similarity_scores)[::-1][:top_n]
    
    # Формируем итоговый список рекомендаций
    recommendations = []
    for idx in top_book_indices:
        recommendations.append({
            'book_id': books_df.iloc[idx]['book_id'],
            'title': books_df.iloc[idx]['title'],
            'cbf_score': similarity_scores[idx] # Оценка близости от 0.0 до 1.0
        })
        
    return pd.DataFrame(recommendations)

In [82]:
active_user_cbf = get_cbf_recommendations(active_user_id, top_n=100)
active_user_cbf.head(10)

,book_id,title,cbf_score
0,30283501,The Last I Love You,0.280492
1,16180680,Utterly Loved,0.231579
2,461587,Do Not Stand At My Grave and Weep,0.213894
3,25052044,Bipolar Cowboy,0.203955
4,24648426,Bipolar Cowboy,0.203955
5,36286613,The Organ's Speech,0.202938
6,28441735,"Stressed, Unstressed: Classic Poems to Ease th...",0.196383
7,6493451,The Octopus Frontier,0.186309
8,26129694,When I Was a Twin,0.179821
9,16249566,Young Americans,0.172718


In [123]:
get_matched_books(active_user_id, active_user_cbf)

,book_id,title,cbf_score,rating,is_read
97,289818,The Nation's Favourite Poems: Book 1,0.105259,4,True


## Collaborative Filtering

In [125]:
reader = Reader(rating_scale=(1, 5))
train_data_surprise = Dataset.load_from_df(
    interactions_train[['user_id', 'book_id', 'rating']], 
    reader
)
trainset = train_data_surprise.build_full_trainset()

user_knn_model = KNNWithMeans(k=40, sim_options={'name': 'cosine', 'user_based': True}, verbose=False)
# user_knn_model.fit(trainset)

item_knn_model = KNNWithMeans(k=40, sim_options={'name': 'cosine', 'user_based': False}, verbose=False)
item_knn_model.fit(trainset)

svd_model = SVD(n_factors=50, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)

def get_cf_prediction(user_id, book_id, model_type='svd'):
    if model_type == 'user_knn':
        return user_knn_model.predict(user_id, book_id).est
    elif model_type == 'item_knn':
        return item_knn_model.predict(user_id, book_id).est
    else:
        return svd_model.predict(user_id, book_id).est

def get_cf_recommendations(user_id, candidate_books, model_type='svd', top_n=10):
    recommendations = []
    for _, row in candidate_books.iterrows():
        b_id = row['book_id']
        
        cf_rating = get_cf_prediction(user_id, b_id, model_type=model_type)
        cf_score = (cf_rating - 1.0) / (5.0 - 1.0)
        
        recommendations.append({
            'book_id': b_id,
            'title': row['title'],
            'cf_score': cf_rating,
            'cf_score_norm': cf_score
        })
        
    recommendations_df = pd.DataFrame(recommendations)
    return recommendations_df.sort_values(by='cf_score_norm', ascending=False).head(top_n).reset_index(drop=True)

In [126]:
active_user_svd = get_cf_recommendations(active_user_id, books_df, top_n=200)
active_user_svd.head(10)

,book_id,title,cf_score,cf_score_norm
0,460853,Withnail and I: the Original Screenplay,4.905810,0.976452
1,1036552,The Riverside Shakespeare,4.877552,0.969388
2,22204746,An Elephant Is On My House: And Other Poems,4.864511,0.966128
3,18459455,"Peter, Enchantment and Stardust: The Poems (Pe...",4.847352,0.961838
4,50683,Uses of the Erotic: The Erotic as Power,4.840388,0.960097
5,1247393,Abol Tabol: The Nonsense World of Sukumar Ray,4.836049,0.959012
6,22047662,Monster Poems for Monstrous Kids (Top of the W...,4.814017,0.953504
7,21911024,Words are our Sorcery,4.808096,0.952024
8,35132873,Vintage Sadness,4.799690,0.949922
9,33375618,Don't Call Us Dead: Poems,4.790651,0.947663


In [127]:
get_matched_books(active_user_id, active_user_svd)

,book_id,title,cf_score,cf_score_norm,rating,is_read
195,197315,Prufrock and Other Observations,4.511986,0.877997,5,True


In [136]:
active_user_item = get_cf_recommendations(active_user_id, books_df, model_type='item_knn', top_n=2000)
active_user_item.head(10)

,book_id,title,cf_score,cf_score_norm
0,9874488,Take Out from the Writer's Café,5.0,1.0
1,18375121,Celebrations In The Ossuary & Other Poems,5.0,1.0
2,32801250,Hamlet,5.0,1.0
3,13518830,The Exercise Book: creative writing exercises ...,5.0,1.0
4,17904575,Kind,5.0,1.0
5,33536938,How the Heartache Humbled Me,5.0,1.0
6,28503633,Pop Sonnets: Shakespearean Spins on Your Favor...,5.0,1.0
7,31934665,Poetry for Kids: Walt Whitman,5.0,1.0
8,1924536,A Preface To Paradise Lost,5.0,1.0
9,747051,The True Subject: Selected Poems,5.0,1.0


In [137]:
get_matched_books(active_user_id, active_user_item)

,book_id,title,cf_score,cf_score_norm,rating,is_read
1141,5635977,The Huntress,4.897857,0.974464,5,True
1803,2963452,Europa,4.652593,0.913148,5,True


## Hybrid Implementation

In [143]:
def get_hybrid_recommendations(user_id, books_df, cf_model_type='svd', alpha=0.7, top_n=10, cold_start_threshold=5):
    # 1. ПЕРЕКЛЮЧАЮЩАЯСЯ СТРАТЕГИЯ (Switching Logic)
    # Извлекаем все прошлые оценки пользователя из обучающей выборки
    user_train_ratings = interactions_train[interactions_train['user_id'] == user_id]
    
    # Сценарий А: Полный холодный старт или критически мало оценок (Переключение на CBF)
    if len(user_train_ratings) < cold_start_threshold:
        cbf_res = get_cbf_recommendations(user_id, top_n=top_n)
        
        # Если у пользователя есть текстовый профиль для работы CBF
        if isinstance(cbf_res, pd.DataFrame) and not cbf_res.empty:
            cbf_res = cbf_res.rename(columns={'cbf_score': 'final_score'})
            cbf_res['strategy'] = 'switching_cold_start_cbf'
            return cbf_res.reset_index(drop=True)
        else:
            # Сценарий Б: Абсолютный холодный старт (Fallback на глобальную популярность)
            popular = books_df.sort_values(by='ratings_count', ascending=False).head(top_n)
            return popular[['book_id', 'title']].assign(final_score=1.0, strategy='fallback_popular').reset_index(drop=True)

    # 2. ВЗВЕШЕННАЯ СТРАТЕГИЯ (Weighted Hybrid Logic)
    # Для генерации рекомендаций берем топ-100 кандидатов от контентной модели для перевзвешивания
    candidate_books = get_cbf_recommendations(user_id, top_n=100)
    
    # Если кандидат пустой (защитный блок кода)
    if not isinstance(candidate_books, pd.DataFrame) or candidate_books.empty:
        popular = books_df.sort_values(by='ratings_count', ascending=False).head(top_n)
        return popular[['book_id', 'title']].assign(final_score=1.0, strategy='fallback_popular').reset_index(drop=True)

    hybrid_scores = []
    for _, row in candidate_books.iterrows():
        b_id = row['book_id']
        cbf_score = row['cbf_score']
        
        # Вызываем нужный коллаборативный модуль через ранее созданный интерфейс (svd или item_knn)
        cf_rating = get_cf_prediction(user_id, b_id, model_type=cf_model_type)
        
        # Мин-макс нормализация оценки коллаборативного фильтра (Surprise выдает от 1.0 до 5.0) к шкале [0.0, 1.0]
        cf_score_norm = (cf_rating - 1.0) / (5.0 - 1.0)
        
        # Формула линейного взвешивания
        final_score = alpha * cf_score_norm + (1.0 - alpha) * cbf_score
        
        hybrid_scores.append({
            'book_id': b_id,
            'title': row['title'],
            'final_score': final_score,
            'strategy': f'weighted_{cf_model_type}_hybrid'
        })
        
    df_hybrid = pd.DataFrame(hybrid_scores)
    # Сортируем по итоговому гибридному весу и отдаем топ-K
    return df_hybrid.sort_values(by='final_score', ascending=False).head(top_n).reset_index(drop=True)


In [144]:
active_user_hybrid = get_hybrid_recommendations(active_user_id, books_df, top_n=100)
active_user_hybrid.head(10)

,book_id,title,final_score,strategy
0,461587,Do Not Stand At My Grave and Weep,0.687008,weighted_svd_hybrid
1,163167,Mostly True: Collected Stories & Drawings,0.676375,weighted_svd_hybrid
2,16180680,Utterly Loved,0.667906,weighted_svd_hybrid
3,24648426,Bipolar Cowboy,0.661620,weighted_svd_hybrid
4,10200029,Some Lamb,0.648257,weighted_svd_hybrid
5,29570442,Landscape with Headless Mama: Poems,0.646780,weighted_svd_hybrid
6,13595971,Collected Poems 1909-1962,0.640946,weighted_svd_hybrid
7,25915454,"Reconsolidation: Or, it's the ghosts who will ...",0.639572,weighted_svd_hybrid
8,13368928,What Gives Us Our Names,0.633954,weighted_svd_hybrid
9,19045757,Voices In My Head,0.629381,weighted_svd_hybrid


In [145]:
get_matched_books(active_user_id, active_user_hybrid)

,book_id,title,final_score,strategy,rating,is_read
24,289818,The Nation's Favourite Poems: Book 1,0.609096,weighted_svd_hybrid,4,True


In [146]:
active_user_hybrid_item = get_hybrid_recommendations(active_user_id, books_df, cf_model_type="item_knn", top_n=100)
active_user_hybrid_item.head(10)

,book_id,title,final_score,strategy
0,30283501,The Last I Love You,0.784148,weighted_item_knn_hybrid
1,21856215,The Tenderness of Mountains,0.747491,weighted_item_knn_hybrid
2,16180680,Utterly Loved,0.745703,weighted_item_knn_hybrid
3,32855529,Revealing Layers: The Shell of a Woman,0.742451,weighted_item_knn_hybrid
4,34209683,Love For Slaughter,0.737595,weighted_item_knn_hybrid
5,19444310,The Parliament of Fowls: In a Modern English V...,0.735366,weighted_item_knn_hybrid
6,26102431,Radical Verses,0.735024,weighted_item_knn_hybrid
7,24724766,Eating My Grandmother: A Grief Cycle,0.734882,weighted_item_knn_hybrid
8,32057973,A Million-Dollar Bill,0.734659,weighted_item_knn_hybrid
9,31748042,The Lovely Brush: Poems,0.733984,weighted_item_knn_hybrid


In [147]:
get_matched_books(active_user_id, active_user_hybrid_item)

,book_id,title,final_score,strategy,rating,is_read
66,289818,The Nation's Favourite Poems: Book 1,0.563769,weighted_item_knn_hybrid,4,True


In [148]:
cold_user_hybrid = get_hybrid_recommendations(cold_user_id, books_df, top_n=200)
cold_user_hybrid.head(10)

,book_id,title,final_score,strategy
0,30119,Where the Sidewalk Ends,1.0,fallback_popular
1,1381,The Odyssey,1.0,fallback_popular
2,1420,Hamlet,1.0,fallback_popular
3,30118,A Light in the Attic,1.0,fallback_popular
4,1371,The Iliad,1.0,fallback_popular
5,23919,The Complete Stories and Poems,1.0,fallback_popular
6,2696,The Canterbury Tales,1.0,fallback_popular
7,2547,The Prophet,1.0,fallback_popular
8,15997,Paradise Lost,1.0,fallback_popular
9,23513349,Milk and Honey,1.0,fallback_popular


In [149]:
get_matched_books(cold_user_id, cold_user_hybrid)

,book_id,title,final_score,strategy,rating,is_read
134,18288210,No Matter the Wreckage,1.0,fallback_popular,5,True


## Metrics

In [ ]:
# Настройки ранжирования
K_VALUE = 5
RELEVANCE_THRESHOLD = 4

# Списки для сбора промежуточных результатов по пользователям
users_test_grouped = interactions_test.groupby('user_id')

# Результирующие словари для хранения финальных метрик по моделям
model_types = ['item_knn', 'svd', 'cbf', 'switching_hybrid', 'weighted_hybrid']
metrics_results = {m: {'mae': [], 'rmse': [], 'precision': [], 'recall': []} for m in model_types}

# Глобальное среднее для подстраховки в расчете ошибок
global_train_mean = interactions_train['rating'].mean()

# Итерационный проход по каждому пользователю из тестовой выборки
for user_id, user_test_df in users_test_grouped:
    actual_books = set(user_test_df['book_id'])
    # Книги, которые пользователь действительно оценил хорошо (Истинные релевантные объекты)
    actual_relevant_books = set(user_test_df[user_test_df['rating'] >= RELEVANCE_THRESHOLD]['book_id'])
    
    # 1. РАСЧЕТ ОШИБОК ПРОГНОЗА (MAE, RMSE)
    actual_ratings_dict = dict(zip(user_test_df['book_id'], user_test_df['rating']))
    
    for b_id, true_r in actual_ratings_dict.items():
        # Сбор прогнозов от коллаборативных моделей
        pred_item_knn = get_cf_prediction(user_id, b_id, model_type='item_knn')
        pred_svd = get_cf_prediction(user_id, b_id, model_type='svd')
        
        metrics_results['item_knn']['mae'].append(abs(true_r - pred_item_knn))
        metrics_results['item_knn']['rmse'].append((true_r - pred_item_knn) ** 2)
        
        metrics_results['svd']['mae'].append(abs(true_r - pred_svd))
        metrics_results['svd']['rmse'].append((true_r - pred_svd) ** 2)
        
        # Для CBF и Гибридов расчет поточечной MAE/RMSE не всегда репрезентативен, 
        # так как они генерируют абстрактные веса релевантности [0,1], а не оценки звезд.
        # Поэтому их эффективность оценивается ниже через качество ранжирования Топ-К списков.
        
    # 2. РАСЧЕТ КАЧЕСТВА РАНЖИРОВАНИЯ ТОП-К СПИСКОВ
    if len(actual_relevant_books) == 0:
        continue  # Пропускаем расчет Precision/Recall, если у пользователя в тесте нет любимых книг
        
    # Генерируем рекомендации от всех доступных моделей
    # а) Рекомендации CF и CBF
    # (Берем топ-100 от CBF в качестве кандидатов и сортируем по оценке Item-KNN для ускорения)
    cbf_candidates = get_cbf_recommendations(user_id, top_n=100)
    if isinstance(cbf_candidates, pd.DataFrame) and not cbf_candidates.empty:
        cbf_candidates['knn_score'] = cbf_candidates['book_id'].apply(lambda x: get_cf_prediction(user_id, x, 'item_knn'))
        rec_item_knn = list(cbf_candidates.sort_values(by='knn_score', ascending=False).head(K_VALUE)['book_id'])
        rec_svd = list(cbf_candidates.assign(svd_score=lambda df: df['book_id'].apply(lambda x: get_cf_prediction(user_id, x, 'svd'))).sort_values(by='svd_score', ascending=False).head(K_VALUE)['book_id'])
        rec_cbf = list(cbf_candidates.head(K_VALUE)['book_id'])
    else:
        rec_item_knn = rec_svd = rec_cbf = []

    # б) Рекомендации Гибридов
    rec_weighted = list(get_hybrid_recommendations(user_id, books_df, interactions_train, cf_model_type='svd', alpha=0.7, top_n=K_VALUE)['book_id'])
    rec_switching = list(get_hybrid_recommendations(user_id, books_df, interactions_train, cf_model_type='svd', alpha=0.7, top_n=K_VALUE, cold_start_threshold=5)['book_id'])
    
    recommendation_lists = {
        'item_knn': rec_item_knn,
        'svd': rec_svd,
        'cbf': rec_cbf,
        'weighted_hybrid': rec_weighted,
        'switching_hybrid': rec_switching
    }
    
    # Расчет метрик пересечения для каждого алгоритма
    for model_name, rec_list in recommendation_lists.items():
        if not rec_list:
            metrics_results[model_name]['precision'].append(0.0)
            metrics_results[model_name]['recall'].append(0.0)
            continue
            
        # Находим книги, которые мы порекомендовали, и пользователь их реально полюбил
        intersection = set(rec_list).intersection(actual_relevant_books)
        
        precision_val = len(intersection) / len(rec_list)
        recall_val = len(intersection) / len(actual_relevant_books)
        
        metrics_results[model_name]['precision'].append(precision_val)
        metrics_results[model_name]['recall'].append(recall_val)

final_rows = []
for model_name in model_types:
    # Расчет метрик предсказания рейтингов
    mae_score = np.mean(metrics_results[model_name]['mae']) if metrics_results[model_name]['mae'] else np.nan
    mse_score = np.mean(metrics_results[model_name]['rmse']) if metrics_results[model_name]['rmse'] else np.nan  # Истинный MSE
    rmse_score = np.sqrt(mse_score) if not np.isnan(mse_score) else np.nan  # Истинный RMSE
    
    # Расчет метрик ранжирования Топ-К
    precision_score = np.mean(metrics_results[model_name]['precision']) if metrics_results[model_name]['precision'] else 0.0
    recall_score = np.mean(metrics_results[model_name]['recall']) if metrics_results[model_name]['recall'] else 0.0
    
    # Расчет комплексной F1-меры в строгом соответствии с формулой из п. 2.2.4
    if precision_score + recall_score > 0:
        f1_score = 2 * (precision_score * recall_score) / (precision_score + recall_score)
    else:
        f1_score = 0.0
        
    final_rows.append({
        'Алгоритм (Модель)': model_name.upper().replace('_', ' '),
        'MAE ↓': f"{mae_score:.3f}" if not np.isnan(mae_score) else "N/A",
        'MSE ↓': f"{mse_score:.3f}" if not np.isnan(mse_score) else "N/A",  # Добавлено для синхронизации с гл. 2
        'RMSE ↓': f"{rmse_score:.3f}" if not np.isnan(rmse_score) else "N/A",
        f'Precision@{K_VALUE} ↑': f"{precision_score:.3f}",
        f'Recall@{K_VALUE} ↑': f"{recall_score:.3f}",
        f'F1-score@{K_VALUE} ↑': f"{f1_score:.3f}"  # Добавлено для синхронизации с гл. 2
    })

df_metrics_table = pd.DataFrame(final_rows)
df_metrics_table

In [150]:
K_VALUE = 5
RELEVANCE_THRESHOLD = 4
global_mean = interactions_train['rating'].mean()

test_pairs = list(interactions_test[['user_id', 'book_id', 'rating']].itertuples(index=False, name=None))

predictions_item_knn = item_knn_model.test(test_pairs)
predictions_svd = svd_model.test(test_pairs)

item_mae = accuracy.mae(predictions_item_knn, verbose=False)
item_rmse = accuracy.rmse(predictions_item_knn, verbose=False)
item_mse = item_rmse ** 2

svd_mae = accuracy.mae(predictions_svd, verbose=False)
svd_rmse = accuracy.rmse(predictions_svd, verbose=False)
svd_mse = svd_rmse ** 2

test_ground_truth = interactions_test[interactions_test['rating'] >= RELEVANCE_THRESHOLD].groupby('user_id')['book_id'].apply(set).to_dict()

train_user_counts = interactions_train['user_id'].value_counts().to_dict()

pred_dict_knn = {u: {} for u in test_ground_truth.keys()}
pred_dict_svd = {u: {} for u in test_ground_truth.keys()}

for uid, iid, _, est, _ in predictions_item_knn:
    if uid in pred_dict_knn: pred_dict_knn[uid][iid] = est
for uid, iid, _, est, _ in predictions_svd:
    if uid in pred_dict_svd: pred_dict_svd[uid][iid] = est

model_names = ['item_knn', 'svd', 'cbf', 'switching_hybrid', 'weighted_hybrid']
final_metrics = {m: {'p': [], 'r': []} for m in model_names}

all_cbf_similarities = cosine_similarity(tfidf_user_matrix, tfidf_book_matrix)

for user_id, actual_relevant_books in test_ground_truth.items():
    if not actual_relevant_books:
        continue
        
    if user_id in user_id_to_idx:
        u_vector_idx = user_id_to_idx[user_id]
        user_cbf_scores = all_cbf_similarities[u_vector_idx]
        top_100_candidate_indices = np.argsort(user_cbf_scores)[::-1][:100]
        
        candidate_book_ids = books_df.iloc[top_100_candidate_indices]['book_id'].values
        candidate_cbf_scores = user_cbf_scores[top_100_candidate_indices]
        cbf_lookup = dict(zip(candidate_book_ids, candidate_cbf_scores))
    else:
        candidate_book_ids = books_df.sort_values(by='ratings_count', ascending=False).head(100)['book_id'].values
        cbf_lookup = {b: 0.0 for b in candidate_book_ids}

    # 1. Ранжирование Топ-5 для CBF
    rec_cbf = candidate_book_ids[:K_VALUE]
    
    # 2. Ранжирование Топ-5 для Item-KNN
    knn_preds = {b: pred_dict_knn[user_id].get(b, global_mean) for b in candidate_book_ids}
    rec_knn = sorted(knn_preds, key=knn_preds.get, reverse=True)[:K_VALUE]
    
    # 3. Ранжирование Топ-5 для SVD
    svd_preds = {b: pred_dict_svd[user_id].get(b, global_mean) for b in candidate_book_ids}
    rec_svd = sorted(svd_preds, key=svd_preds.get, reverse=True)[:K_VALUE]
    
    # 4. Ранжирование Топ-5 для Взвешенного гибрида
    weighted_scores = {}
    for b in candidate_book_ids:
        c_score = cbf_lookup.get(b, 0.0)
        s_rating = svd_preds.get(b, global_mean)
        s_score_norm = (s_rating - 1.0) / (5.0 - 1.0)
        weighted_scores[b] = 0.7 * s_score_norm + 0.3 * c_score
    rec_weighted = sorted(weighted_scores, key=weighted_scores.get, reverse=True)[:K_VALUE]
    
    # 5. Ранжирование Топ-5 для Переключающегося гибрида
    user_train_count = train_user_counts.get(user_id, 0)
    rec_switching = rec_cbf if user_train_count < 5 else rec_weighted
    
    lists = {'item_knn': rec_knn, 'svd': rec_svd, 'cbf': rec_cbf, 'switching_hybrid': rec_switching, 'weighted_hybrid': rec_weighted}
    for m_name, r_list in lists.items():
        intersection = set(r_list).intersection(actual_relevant_books)
        p = len(intersection) / len(r_list) if len(r_list) > 0 else 0
        r = len(intersection) / len(actual_relevant_books) if len(actual_relevant_books) > 0 else 0
        final_metrics[m_name]['p'].append(p)
        final_metrics[m_name]['r'].append(r)

In [153]:
final_rows = []
for m_name in model_names:
    p_mean = np.mean(final_metrics[m_name]['p'])
    r_mean = np.mean(final_metrics[m_name]['r'])
    f1_mean = 2 * (p_mean * r_mean) / (p_mean + r_mean) if (p_mean + r_mean) > 0 else 0
    
    if m_name == 'item_knn': mae, mse, rmse = item_mae, item_mse, item_rmse
    elif m_name == 'svd': mae, mse, rmse = svd_mae, svd_mse, svd_rmse
    else: mae, mse, rmse = np.nan, np.nan, np.nan

    if m_name == 'item_knn': m_name = "item-based CF"
    final_rows.append({
        'Алгоритм (Модель)': m_name.upper().replace('_', ' '),
        'MAE ↓': f"{mae:.4f}" if not np.isnan(mae) else "N/A",
        'MSE ↓': f"{mse:.4f}" if not np.isnan(mse) else "N/A",
        'RMSE ↓': f"{rmse:.4f}" if not np.isnan(rmse) else "N/A",
        'Precision@5 ↑': f"{p_mean:.4f}",
        'Recall@5 ↑': f"{r_mean:.4f}",
        'F1-score@5 ↑': f"{f1_mean:.4f}"
    })

df_metrics_table = pd.DataFrame(final_rows)
df_metrics_table

,Алгоритм (Модель),MAE ↓,MSE ↓,RMSE ↓,Precision@5 ↑,Recall@5 ↑,F1-score@5 ↑
0,ITEM-BASED CF,0.6909,0.8440,0.9187,0.1064,0.3844,0.1667
1,SVD,0.6951,0.7577,0.8705,0.1084,0.3977,0.1703
2,CBF,N/A,N/A,N/A,0.0797,0.3260,0.1281
3,SWITCHING HYBRID,N/A,N/A,N/A,0.0985,0.3620,0.1548
4,WEIGHTED HYBRID,N/A,N/A,N/A,0.1082,0.3973,0.1700
